# Met Eyes

# Get Data

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data

In [ ]:
import json
import requests

from os import listdir, makedirs, path
from PIL import Image as PImage
from time import sleep

from utils import export_combined_jsons

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

JSON_OBJS_DIR = f"{JSON_DIR}/objects"
JSON_FACES_DIR = f"{JSON_DIR}/faces"
JSON_LANDMARKS_DIR = f"{JSON_DIR}/landmarks"

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
makedirs(IMG_DIR, exist_ok=True)
makedirs(JSON_DIR, exist_ok=True)
makedirs(JSON_OBJS_DIR, exist_ok=True)
makedirs(JSON_FACES_DIR, exist_ok=True)
makedirs(JSON_LANDMARKS_DIR, exist_ok=True)

In [ ]:
MET_URL = "https://collectionapi.metmuseum.org/public/collection/v1"

yolo_model_path = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
face_detector = YOLO(yolo_model_path)

landmarker_model_path = "./face_landmarker.task"

landmarker_options = mpFaceLandmarkerOptions(
  base_options=mpBaseOptions(model_asset_path=landmarker_model_path),
  running_mode=mpRunningMode.IMAGE
)

landmarker = mpFaceLandmarker.create_from_options(landmarker_options)

### Get Object IDs

- $14\text{,}982$ paintings available in API
- $14\text{,}200$ have images

In [ ]:
obj_ids = []

collection_response = requests.get(f"{MET_URL}/search?medium=Paintings&hasImages=true&q=*")
query_obj_ids = set(collection_response.json()["objectIDs"])
obj_ids += list(query_obj_ids)

len(obj_ids)

### Get Object Metadata

In [ ]:
obj_fields = [
  "objectID",
  "accessionNumber",
  "objectName",
  "title",
  "department",
  "primaryImage",
  "artistRole",
  "artistDisplayName",
  "objectDate",
  "objectBeginDate",
  "objectEndDate",
  "medium",
  "dimensions"
]

def get_measurement(obj_data):
  if "measurements" in obj_data and len(obj_data["measurements"]) > 0:
    img_meas = [m["elementMeasurements"] for m in obj_data["measurements"] if m["elementName"] == "Image"]
    ovr_meas = [m["elementMeasurements"] for m in obj_data["measurements"] if m["elementName"] == "Overall"]
    otr_meas = [m["elementMeasurements"] for m in obj_data["measurements"] if m["elementName"] == "Other"]
    smt_meas = [m["elementMeasurements"] for m in obj_data["measurements"] if "Height" in m["elementMeasurements"] and "Width" in m["elementMeasurements"]]

    if len(img_meas) > 0:
      return img_meas[0]
    elif len(ovr_meas) > 0:
      return ovr_meas[0]
    elif len(otr_meas) > 0:
      return otr_meas[0]
    elif len(smt_meas) > 0:
      return smt_meas[0]
    else:
      return None
  else:
    return None


def get_obj_data(oid):
  json_obj_path = f"{JSON_OBJS_DIR}/{oid}.json"
  try:
    with open(json_obj_path, "r") as ifp:
      return json.load(ifp)
  except FileNotFoundError:
    obj_response = requests.get(f"{MET_URL}/objects/{oid}")
    obj_data = obj_response.json()

    if not ("primaryImage" in obj_data and obj_data["primaryImage"].startswith("http")):
      return None

    obj_filtered_data = { f: obj_data[f] for f in obj_fields }

    obj_measurements = get_measurement(obj_data)
    if obj_measurements:
      obj_filtered_data["measurements"] = obj_measurements

    if "tags" in obj_data and obj_data["tags"] and len(obj_data["tags"]) > 0:
      obj_filtered_data["tags"] = [t["term"].lower() for t in obj_data["tags"]]

    with open(json_obj_path, "w") as ofp:
      json.dump(obj_filtered_data, ofp, ensure_ascii=False)
    return obj_filtered_data

In [ ]:
def get_face_data(obj_data, img):
  obj_data = json.loads(json.dumps(obj_data))
  oid = obj_data["objectID"]
  json_face_path = f"{JSON_FACES_DIR}/{oid}.json"
  try:
    with open(json_face_path, "r") as ifp:
      return json.load(ifp)
  except FileNotFoundError:
    iw,ih = img.size
    nh = 256
    nw = int(nh * iw // ih)
    nimg = img.resize((nw, nh))

    faces = face_detector.predict(nimg, verbose=False, device="cuda")
    if len(faces) < 1 or len(faces[0]) < 1:
      obj_data["faces"] = {
        "yolo": {
          "count": 0,
          "xyxyn": [],
          "xyxyn_sq": [],
        }
      }
      return obj_data

    faces_xyxyn = faces[0].boxes.xyxyn.cpu().numpy().astype(np.float64)
    faces_xyxyn_sq = pcts_to_sqs(faces_xyxyn, iw, ih)

    obj_data["faces"] = {
      "yolo": {
        "count": len(faces_xyxyn),
        "xyxyn": faces_xyxyn.round(4).tolist(),
        "xyxyn_sq": faces_xyxyn_sq.round(4).tolist(),
      }
    }

    with open(face_json_path, "w") as ofp:
      json.dump(obj_data, ofp, ensure_ascii=False)
    return obj_data

In [ ]:
def get_landmark_data(obj_data, img):
  obj_data = json.loads(json.dumps(obj_data))
  oid = obj_data["objectID"]
  json_landmark_path = f"{JSON_LANDMARKS_DIR}/{oid}.json"
  try:
    with open(json_landmark_path, "r") as ifp:
      return json.load(ifp)
  except FileNotFoundError:
    iw,ih = img.size
    mp_results = {
      "count": 0,
      "landmarks": [],
    }

    for fcnt,fbox in enumerate(obj_data["faces"]["yolo"]["xyxyn_sq"]):
      x0,y0,x1,y1 = pct_to_px(fbox, iw, ih)
      fx0,fy0,fx1,fy1 = fbox
      fw, fh = (fx1 - fx0), (fy1 - fy0)

      fimg = img.crop((x0,y0,x1,y1)).resize((512, 512)).convert("RGB")
      mp_image = mpImage(image_format=mpImageFormat.SRGB, data=np.array(fimg))
      landmarks = landmarker.detect(mp_image)

      face_landmarks = np.array([]).astype(np.float64)

      if len(landmarks.face_landmarks) > 0:
        mp_results["count"] += 1
        face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)
      else:
        fimg = fimg.resize((128, 128))
        mp_image = mpImage(image_format=mpImageFormat.SRGB, data=np.array(fimg))
        landmarks = landmarker.detect(mp_image)

        if len(landmarks.face_landmarks) > 0:
          mp_results["count"] += 1
          face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)
        else:
          fimg = fimg.resize((64, 64))
          mp_image = mpImage(image_format=mpImageFormat.SRGB, data=np.array(fimg))
          landmarks = landmarker.detect(mp_image)

          if len(landmarks.face_landmarks) > 0:
            mp_results["count"] += 1
            face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)

      mp_results["landmarks"].append(face_landmarks.round(4).tolist())

    obj_data["faces"]["mp"] = mp_results

    with open(json_landmark_path, "w") as ofp:
      json.dump(obj_data, ofp, ensure_ascii=False)
    return obj_data

In [ ]:
def save_eye_pairs(obj_data, img):
  obj_data = json.loads(json.dumps(obj_data))
  oid = obj_data["objectID"]
  iw,ih = img.size
  # TODO: for loop
  #   TODO: check file
  #   TODO: extract and save

In [ ]:
for cnt,oid in enumerate(obj_ids[:20]):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  obj_data = get_obj_data(oid)

  if obj_data is None:
    continue

  img_url = obj_data["primaryImage"]
  img_response = requests.get(img_url, stream=True)
  img = PImage.open(img_response.raw)

  face_data = get_face_data(obj_data, img)

  landmark_data = get_landmark_data(face_data, img)

  # TODO: crop eyes from img and save avif
  save_eye_pairs(landmark_data, img)

  sleep(0.1)

### Export Combined Object Metadata

In [ ]:
export_combined_jsons(JSON_OBJS_DIR, JSON_DIR, "objects")
export_combined_jsons(JSON_FACES_DIR, JSON_DIR, "faces")
export_combined_jsons(JSON_LANDMARK_DIR, JSON_DIR, "landmarks")